In [ ]:
import os
import re
import glob
import copy
import enum
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

In [ ]:
input_folder = "../build/cutsPower__2023-08-07T23-59-34"

In [ ]:
col_cut_type = "Cut type"
col_lp_diff = "LP diff"
col_milp_diff = "MILP diff"
col_cutoff_diff = "Cutoff diff"

In [ ]:
def parse_file(filename, problem_name, pattern_list, source_dict):
    with open(filename, "r") as file:
        for line in file:
            # if not line.strip(): 
            #     continue
            for pattern, group_name, column_name in pattern_list:
                match = re.search(pattern, line)
                if match:
                    found = match.groupdict()[group_name]
                    source_dict[column_name].append(found)

In [ ]:
def parse_logs(logs_dir: str):
    problem_dataframes = {}
    # note: regex used for number with exponent: [-+]*\d*\.*\d+([eE][-+]?\d+)?
    pattern_cut_type = r"Cut-type =\s*\"(?P<cut_type>[^\"]+(?=\":))"
    pattern_lp_diff = r"LP Diff. =\s*(?P<lp_diff>[-+]*\d*\.*\d+([eE][-+]?\d+)?)"
    pattern_milp_diff = r"MILP Diff. =\s*(?P<milp_diff>[-+]*\d*\.*\d+([eE][-+]?\d+)?)"
    pattern_cutoff_diff = r"Cutoff Diff. =\s*(?P<cutoff_diff>[-+]*\d*\.*\d+([eE][-+]?\d+)?)"

    pattern_list_cuts = (
        (pattern_cut_type, "cut_type", col_cut_type),
        (pattern_lp_diff, "lp_diff", col_lp_diff),
        (pattern_milp_diff, "milp_diff", col_milp_diff),
        (pattern_cutoff_diff, "cutoff_diff", col_cutoff_diff)
    )

    for filename in glob.glob(f"{logs_dir}/*gz_results.txt"):
        match = re.search(r"(?<=/)[^/]+(?=.mps.gz_results.txt)", filename)
        problem_name = match[0]

        source_dict = defaultdict(list)
        parse_file(filename, problem_name, pattern_list_cuts, source_dict)
        result_df = pd.DataFrame(source_dict)
        for diff in (col_lp_diff, col_milp_diff, col_cutoff_diff):
            if diff in result_df.columns:
                result_df[diff] = result_df[diff].astype(float)
        problem_dataframes[problem_name] = result_df
    
    return problem_dataframes

result_dict = parse_logs(input_folder)
for name, df in result_dict.items():
    print(f"Problem name: {name}")
    print(df)
    print()

In [ ]:
# List of parsed problems
result_dict.keys()

In [ ]:
class diff(enum.Enum):
    lp_diff = col_lp_diff
    milp_diff = col_milp_diff
    cutoff_diff = col_cutoff_diff

In [ ]:
def filter_data(result_dict, target_diff_column):
    for problem, data in list(result_dict.items()):
        if target_diff_column in data.columns:
            data = data.loc[(data[[target_diff_column]] != 0.0).all(axis=1), :]
            result_dict[problem] = data
            if not len(data):
                del result_dict[problem]
        else:
            del result_dict[problem]

def plot_report(result_dict_origin, target_diff=diff.lp_diff, hide_empty_columns=True, zero_none_cuts=False):
    """
    Plot plots for all problems from result_dict_origin for cuts: all, none, clique, my, my+all.
    For every problem, 2 charts are generated:
        * Bar chart which compare absolute value for every cut type
        * Bar chart which compare cut types difference in percent between none cut type and others.
          This chart is not drawn if none cut for current problem is None of 0
    Arguments:
        result_dict_origin: source dict with problem names as keys and dataframes with results as values
        target_diff: value of diff enum, what diff use: LP, MILP or Cutoff
        hide_empty_columns: if set to True, bars for cuts with empty values will be hidden
        zero_none_cuts: used when percent values are calculated. Ff True, none cuts will be considered as 0%, else 100%
    """
    result_dict = copy.deepcopy(result_dict_origin)
    target_diff_column = target_diff.value
    if hide_empty_columns:
        filter_data(result_dict, target_diff_column)

    if not len(result_dict.keys()):
        print(f"There are no problems for {target_diff}")
        return
    
    cuts =   ["all(force)", "all",   "my(split)",  "my(mir)",       "my(split)+all", "my(mir)+all", "none",  "clique", "gomory", "mir",  "liftp", "zeroh", "disj", "gub",  "cover", "flowc", "flowp", "mcf",  "impliedl", "impliedg"]
    colors = ["green",   "green", "red",        "red",           "green",         "green",         "black", "blue",   "blue",   "blue", "blue",  "blue",  "blue", "blue", "blue",  "blue",  "blue",  "blue", "blue",     "blue"]
    color_map = dict(zip(cuts, colors))

    colsize = 2
    coordinates = range(len(cuts))
    data_list = list(result_dict.items())
    none_cut = "none"
    margin = 0.1
    mantissa_len = 4
    text_offset_percent = 0.01   

    fig = plt.figure(constrained_layout=True, figsize=(35, 4 * len(result_dict.keys())))  # figsize=(width, height)
    fig.suptitle('Comparison of different cut types', fontsize='x-large', fontweight='bold')
    subfigs = fig.subfigures(nrows=len(result_dict.keys()), ncols=1, hspace=0.1)
    if len(result_dict) <= 1:
        subfigs = [subfigs]
    for row, subfig in enumerate(subfigs):
        problem_name, data = data_list[row]
        
        # Check if target cut type is presented in data (== cut type was presented in result file)
        if target_diff_column not in data.columns:
            continue

        cut_diff = data.loc[data[col_cut_type].isin(cuts)]
        cut_diff.reset_index(drop=True, inplace=True)
        coordinates = range(len(cut_diff))
        row_axs = subfig.subplots(nrows=1, ncols=colsize)

        # Diff Absolute
        ax = row_axs[0]
        lp_diff_colors = [color_map[c] for c in cut_diff[col_cut_type]]
        ax.bar(coordinates, cut_diff[target_diff_column].abs(), width=0.4, color=lp_diff_colors)  
        ax.set_xticks(coordinates)
        ax.set_xticklabels(cut_diff[col_cut_type])
        ax.set_title(target_diff_column + " absolute")
        for i, coordinate in enumerate(coordinates):
            ax.text(
                coordinate,
                cut_diff[target_diff_column][i] * (1 + text_offset_percent),
                cut_diff[target_diff_column][i],  # round(cut_diff[target_diff_column][i], mantissa_len),
                ha = 'center',
                # bbox = dict(facecolor = 'red', alpha =.7)
            )
        ax.margins(margin)

        # Diff Percent
        none_diff = cut_diff.loc[cut_diff[col_cut_type] == none_cut]
        if len(none_diff):
            none_diff = none_diff.iloc[0][target_diff_column]
        else:
            none_diff = None
        ax = row_axs[1]
        if none_diff:
            none_cut_percent = 0.2 if zero_none_cuts else 100  # in %
            diff_persent = cut_diff[target_diff_column].apply(lambda x: ((abs(x) - abs(none_diff)) / abs(none_diff)) * 100 + none_cut_percent)
            lp_percent_colors = [color_map[c] for c in cut_diff[col_cut_type]]
            ax.bar(coordinates, diff_persent, width=0.4, color=lp_percent_colors)
            ax.set_xticks(coordinates)
            ax.set_xticklabels(cut_diff[col_cut_type])
            ax.set_title(target_diff_column + " %")
            for i, coordinate in enumerate(coordinates):
                current_percent_diff = round(diff_persent[i] - none_cut_percent, mantissa_len)
                sign = ""
                if current_percent_diff < 0:
                    sign = "- "
                elif current_percent_diff > 0:
                    sign = "+ "
                label = sign + str(abs(current_percent_diff)) + "%"
                ax.text(
                    coordinate,
                    diff_persent[i] * (1 + text_offset_percent),
                    label,
                    ha = 'center',
                    # bbox = dict(facecolor = 'red', alpha =.7)
                )
            ax.margins(margin)
        else:
            # Do not plot in case if none cut diff is not presented in dataframe
            ax.axis('off')

        subfig.suptitle(problem_name)
    plt.show()

#### Plot for LP diff

In [ ]:
plot_report(result_dict, diff.lp_diff)

#### Plot for Cutoff diff

In [ ]:
# plot_report(result_dict, diff.cutoff_diff)